# KernelFuse Phase 6 — vLLM dry run (Tier B / T4)

**Exit gate:** met on Colab T4 — see repo `docs/phase_6_report.md`.

This notebook remains the reference path. Use a **fresh zip/clone** of the repo (tokenizer prompts + `--smoke`).

T4: `--dtype float16`, expect **TRITON_ATTN** (not FA), `enforce_eager` for dry run only.
Install cell → **Runtime → Restart** → then serve/harness.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## Cell A — install only

RAPIDS/cuml pip conflict warnings are Colab noise. Then **Runtime → Restart session**.

In [ ]:
!pip -q install -U "vllm" "openai" "pyyaml" "huggingface_hub" "transformers"
print("Install done. Runtime → Restart session, then Cell B.")

## Cell B — repo + HF_TOKEN secret

In [ ]:
import os, shutil, zipfile
from pathlib import Path
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        print("HF_TOKEN from secrets")
except Exception as e:
    print("no HF_TOKEN secret:", e)

ROOT = Path("/content/KernelFuse")
def has_sources(root: Path) -> bool:
    return (root / "bench/runner.py").is_file() and (root / "bench/prompts.py").is_file()

zpath = next((z for z in [Path("/content/kernelfuse_upload.zip"), Path("/content/phase6_upload.zip")] if z.is_file()), None)
if not has_sources(ROOT):
    if zpath:
        if ROOT.exists(): shutil.rmtree(ROOT)
        with zipfile.ZipFile(zpath) as zf: zf.extractall("/content")
    else:
        if ROOT.exists(): shutil.rmtree(ROOT)
        !git clone --depth 1 https://github.com/KashMaj1708/KernelFuse.git {ROOT}
os.chdir(ROOT)
assert has_sources(ROOT), "Need fresh Phase 6+ tree (bench/prompts.py)"
import vllm, torch
print("ok", ROOT, "vllm", vllm.__version__, "torch", torch.__version__)

## Start vLLM (self-contained; v0.27 flags)

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path
ROOT = Path("/content/KernelFuse"); os.chdir(ROOT)
!pip uninstall -y torchaudio -q
shim_root = Path("/content/torchaudio_shim")
(shim_root / "torchaudio").mkdir(parents=True, exist_ok=True)
(shim_root / "torchaudio" / "__init__.py").write_text(
    '__version__="0.0.0+shim"\nclass AudioMetaData: pass\ndef load(*a,**k): raise RuntimeError("shim")\ndef save(*a,**k): raise RuntimeError("shim")\n'
)
!fuser -k 8000/tcp 2>/dev/null || true
time.sleep(1)
env = os.environ.copy()
env["PYTHONPATH"] = str(shim_root) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["VLLM_LOGGING_LEVEL"] = "INFO"
log_path = Path("/content/vllm_server.log")
cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
       "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "--host", "0.0.0.0", "--port", "8000",
       "--dtype", "float16", "--gpu-memory-utilization", "0.75", "--max-model-len", "2048",
       "--enforce-eager", "--no-enable-log-requests"]
logf = open(log_path, "w", buffering=1)
proc = subprocess.Popen(cmd, cwd=str(ROOT), env=env, stdout=logf, stderr=subprocess.STDOUT)
print("pid", proc.pid)
url = "http://127.0.0.1:8000/v1/models"
for i in range(120):
    if proc.poll() is not None:
        print(log_path.read_text()[-8000:]); raise SystemExit(proc.returncode)
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            print("SERVER READY", r.read()[:200]); break
    except Exception:
        time.sleep(5)
else:
    raise SystemExit("timeout")
text = log_path.read_text().lower()
attn = "triton_attn" if "triton_attn" in text else ("xformers" if "xformers" in text else "unknown")
Path("/content/attention_backend.txt").write_text(attn)
print("attention_backend", attn)

## Harness smoke (current runner + tokenizer prompts)

In [ ]:
import os
from pathlib import Path
ROOT = Path("/content/KernelFuse"); os.chdir(ROOT)
os.environ["KERNELFUSE_VLLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
attn = Path("/content/attention_backend.txt").read_text().strip() if Path("/content/attention_backend.txt").is_file() else "triton_attn"
out = ROOT / "reports/phase6/results_phase6_v1_colab_smoke.csv"
!python -m bench.runner --matrix bench/config_matrix_phase6_v1.yaml --backends vllm --smoke --out {out} --attention-backend {attn}
!head -n 5 {out}
!cat {out.parent}/run_metadata.json